# Zerlaut interactive simulation plots

## Initialization

**Environment**

In [ ]:
from dotenv import dotenv_values, find_dotenv
from pathlib import Path
import os, sys

# Busqueda de ficheros dotenv
defaults_file = find_dotenv('.env_default', usecwd=True)

if not defaults_file:
    raise FileNotFoundError('Start Jupyter within the repository containing .env_default.')
env_dir = Path(defaults_file).resolve().parent

# Actualización de varaibles de entorno
settings = {**dotenv_values(defaults_file), **dotenv_values(env_dir / '.env')}
settings = {k: v if v is not None else '' for k, v in settings.items() }

os.environ.update(settings)

# Librerías auxiliares
import_dir = Path(settings.get('IMPORT_DIR') or env_dir).expanduser()
if not import_dir.is_absolute():
    import_dir = env_dir / import_dir
sys.path.insert(0, str(import_dir.resolve()))

**Modules**

In [ ]:
import json5 as json
import numpy as np
import bqplot as bq
import ipywidgets as widgets
from ipyevents import Event
from IPython.display import display
from scipy.stats import beta, truncnorm
from html import escape

In [ ]:
from aux.environment import resolve_directory
from aux.parameters import flatten_params
from aux.tvb.stimulus import build_stimulus

**Set main directory paths**

In [ ]:
WORK_DIR = resolve_directory(settings['WORK_DIR'], env_dir)

## Load and plot simulations

Load or remove trials using the controls, up to 16 at a time. Hover over a preview to enlarge it and inspect its parameters on the right. The kernel must remain running for these interactions.

All signals use one sample out of every 8. Parameter bars show `(value - default) / (max - min)` as a percentage, using the defaults from the JSON. Their labels show the actual values. Hover over a bar for the default, difference and saved sampling density. Colors show each saved marginal density divided by its maximum density, from 0 to 1; these are densities, not discrete probabilities. Fixed parameters have zero deviation and relative density 1.

In [ ]:
OUTPUT_DIR = WORK_DIR / "results" / "exp1"
TRIAL_IDS = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15] #201
CONFIG_PATH = OUTPUT_DIR / "config.json"
SKIP_VALUES = 8

# PLOT_STATES = None
PLOT_STATES = ["E", "I"]

with CONFIG_PATH.open(encoding="utf-8") as config_file:
    loaded_config = json.load(config_file)

MAX_PLOTS = 16

## Parameter reference

In [ ]:
parameter_config = flatten_params(loaded_config["model"]["sampled_parameters"])
parameter_config |= flatten_params({
    name: {"default": value, "min": value, "max": value}
    for name, value in loaded_config["model"]["fixed_parameters"].items()
})
parameter_names = list(parameter_config)
defaults = np.array([parameter_config[name]["default"] for name in parameter_names], dtype=float)
parameter_ranges = np.array([
    parameter_config[name]["max"] - parameter_config[name]["min"] for name in parameter_names
])

maximum_densities = []
for parameter in parameter_config.values():
    minimum, maximum = parameter["min"], parameter["max"]
    if minimum == maximum:
        density = 1.0
    elif parameter.get("method", "uniform") == "beta":
        peak = (parameter["default"] - minimum) / (maximum - minimum)
        a, b = 1 + peak * parameter["lambda"], 1 + (1 - peak) * parameter["lambda"]
        density = beta.pdf(parameter["default"], a, b, loc=minimum, scale=maximum - minimum)
    elif parameter.get("method", "uniform") == "truncnorm":
        peak, std = parameter["default"], parameter["std"]
        density = truncnorm.pdf(np.clip(peak, minimum, maximum),
                                (minimum - peak) / std, (maximum - peak) / std,
                                loc=peak, scale=std)
    else:
        density = 1.0 / (maximum - minimum)
    maximum_densities.append(density)
maximum_densities = np.asarray(maximum_densities)

## Trial loading and figures

In [ ]:
STATE_UNITS = {
    "E": "kHz", "I": "kHz",
    "C_ee": "kHz²", "C_ei": "kHz²", "C_ii": "kHz²",
    "W_e": "pA", "W_i": "pA", "ou_drift": "N.A.",
}


def load_trial(simulation_id):
    with np.load(OUTPUT_DIR / f"trial_{simulation_id:08d}.npz", allow_pickle=False) as saved:
        time_ms = np.asarray(saved["time_ms"])[::SKIP_VALUES]
        names = (PLOT_STATES if PLOT_STATES is not None else
                 [name for name in saved.files if name not in ("time_ms", "metadata_json")])
        states = {name: np.asarray(saved[name])[::SKIP_VALUES] for name in names}
        metadata = json.loads(saved["metadata_json"].item())

    stimulus_time, stimulus = build_stimulus(
        metadata["stimulus"], loaded_config["simulation"]["duration_ms"],
        loaded_config["simulation"]["dt_ms"], seed=metadata["stimulus_seed"],
    )
    stimulus = {name: signal.ravel()[::SKIP_VALUES] for name, signal in stimulus}
    if loaded_config["model"]["name"] == "ZerlautAdaptationFirstOrder" and "S_i" in metadata["parameters"]:
        if "external_input_in_ex" in stimulus:
            stimulus["external_input_in_ex"] = stimulus["external_input_in_ex"] * metadata["parameters"]["S_i"]

    groups = []
    if stimulus:
        groups.append(("Stimuli", "kHz", stimulus_time[::SKIP_VALUES], stimulus))
    for unit in dict.fromkeys(STATE_UNITS.get(name, "model units") for name in states):
        signals = {name: values for name, values in states.items()
                   if STATE_UNITS.get(name, "model units") == unit}
        groups.append(("State variables", unit, time_ms, signals))

    flat_values = flatten_params({name: {"value": value}
                                 for name, value in metadata["parameters"].items()})
    flat_densities = flatten_params({name: {"value": value}
                                    for name, value in metadata["probabilities"].items()})
    values = np.array([flat_values[name]["value"] for name in parameter_names])
    densities = np.array([flat_densities.get(name, {"value": 1.0})["value"]
                          for name in parameter_names])
    deviations = np.divide(values - defaults, parameter_ranges,
                           out=np.zeros_like(defaults), where=parameter_ranges != 0) * 100
    return {"id": simulation_id, "groups": groups, "values": values,
            "densities": densities, "deviations": deviations,
            "colors": np.clip(densities / maximum_densities, 0, 1), "metadata": metadata}


def make_trial_plot(trial, height, preview=False):
    figures = []
    for kind, unit, time_ms, signals in trial["groups"]:
        xs = bq.LinearScale(min=float(time_ms[0]), max=float(time_ms[-1]))
        ys = bq.LinearScale()
        lines = bq.Lines(x=time_ms, y=np.array(list(signals.values())),
                         scales={"x": xs, "y": ys}, labels=list(signals),
                         display_legend=not preview, tooltip=bq.Tooltip(fields=["name"]))
        figures.append(bq.Figure(
            marks=[lines],
            title=f"{kind} ({unit})" if preview else "",
            title_style={"font-size": "11px"},
            axes=[bq.Axis(scale=xs, label="" if preview else "Time (ms)", num_ticks=3 if preview else 5,
                          tick_style={"font-size": "9px" if preview else "11px"}),
                  bq.Axis(scale=ys, orientation="vertical", label="" if preview else f"{kind} ({unit})",
                          num_ticks=3 if preview else 5,
                          tick_style={"font-size": "9px" if preview else "11px"})],
            fig_margin={"top": 25 if preview else 20, "bottom": 25 if preview else 40,
                        "left": 45 if preview else 60, "right": 25},
            legend_location="top-left", animation_duration=0,
            layout=widgets.Layout(width="100%", height=f"{height}px", min_width="0px", flex="0 0 auto"),
        ))
    return widgets.VBox(figures, layout=widgets.Layout(width="100%", min_width="0px"))

## Interactive view

In [ ]:
trials, previews, hover_events = {}, {}, {}
active_id = None
trial_input = widgets.BoundedIntText(value=TRIAL_IDS[0] if TRIAL_IDS else 0,
                                    min=0, max=2**31 - 1, description="Trial ID")
load_button = widgets.Button(description="Load trial", icon="plus")
loaded_selector = widgets.Dropdown(description="Loaded", options=[])
remove_button = widgets.Button(description="Remove trial", icon="minus", disabled=True)
status = widgets.HTML()
focus_title = widgets.HTML("<b>Hover over a trial to inspect it.</b>")
focus_plot = widgets.VBox(layout=widgets.Layout(width="60%", min_width="0px"))
parameter_tooltip = widgets.HTML()
parameter_heading = widgets.HTML("<b>Parameters vs JSON defaults</b>")
parameter_labels = bq.OrdinalScale(reverse=True)
deviation_scale = bq.LinearScale(min=-100, max=100)
density_scale = bq.ColorScale(min=0, max=1, scheme="Viridis")
parameter_bars = bq.Bars(
    x=[], y=[], color=[], orientation="horizontal",
    scales={"x": parameter_labels, "y": deviation_scale, "color": density_scale},
    tooltip=parameter_tooltip,
)
parameter_figure = bq.Figure(
    marks=[parameter_bars],
    axes=[bq.Axis(scale=parameter_labels, orientation="vertical", side="left", tick_style={"font-size": "10px"}),
          bq.Axis(scale=deviation_scale, label="Difference / sampling range (%)"),
          bq.ColorAxis(scale=density_scale, side="top", label="Relative sampling density",
                       num_ticks=3, tick_format=".1f", label_offset="35px")],
    fig_margin={"top": 95, "bottom": 55, "left": 180, "right": 15},
    animation_duration=0,
    layout=widgets.Layout(width="100%", height=f"{max(450, 23 * len(parameter_names) + 150)}px", flex="0 0 auto"),
)
parameter_panel = widgets.VBox([parameter_heading, parameter_figure],
                              layout=widgets.Layout(width="40%", height="650px", overflow="auto"))
grid = widgets.GridBox(layout=widgets.Layout(
    grid_template_columns="repeat(4, minmax(0, 1fr))", grid_gap="8px",
    width="100%", max_height="340px", overflow="auto",
))


def show_trial(simulation_id):
    global active_id
    if simulation_id == active_id:
        return
    active_id = simulation_id
    trial = trials[simulation_id]
    loaded_selector.value = simulation_id
    focus_title.value = (f"<b>Trial {simulation_id}</b> — "
                         f"joint sampling density: {trial['metadata']['probability']:.4g}")
    if not focus_plot.children:
        focus_plot.children = make_trial_plot(trial, 300).children
    else:
        for figure, (_, _, time_ms, signals) in zip(focus_plot.children, trial["groups"]):
            with figure.marks[0].hold_sync():
                figure.marks[0].x = time_ms
                figure.marks[0].y = np.array(list(signals.values()))
    with parameter_bars.hold_sync():
        parameter_bars.x = [f"{name} = {value:.5g}" for name, value in zip(parameter_names, trial["values"])]
        parameter_bars.y = trial["deviations"]
        parameter_bars.color = trial["colors"]
    parameter_heading.value = f"<b>Trial {simulation_id}: parameters vs defaults</b>"


def show_parameter(mark, event):
    index = event["data"]["index"]
    trial = trials[active_id]
    parameter_tooltip.value = (
        f"<b>{escape(parameter_names[index])}</b><br>Value: {trial['values'][index]:.6g}"
        f"<br>JSON default: {defaults[index]:.6g}"
        f"<br>Difference: {trial['values'][index] - defaults[index]:+.6g}"
        f"<br>Sampling density: {trial['densities'][index]:.6g}"
        f"<br>Relative density: {trial['colors'][index]:.3f}"
    )


def add_trial(simulation_id):
    if simulation_id in trials:
        show_trial(simulation_id)
        return
    if len(trials) == MAX_PLOTS:
        status.value = f"Remove a trial before loading another (maximum {MAX_PLOTS})."
        return
    try:
        trial = load_trial(simulation_id)
    except FileNotFoundError:
        status.value = f"Trial {simulation_id} was not found in {escape(str(OUTPUT_DIR))}."
        return
    trials[simulation_id] = trial
    plot = make_trial_plot(trial, 125, preview=True)
    card = widgets.VBox([widgets.HTML(f"<b>Trial {simulation_id}</b>"), plot],
                        layout=widgets.Layout(border="1px solid #ddd", padding="4px", min_width="0px"))
    listener = Event(source=card, watched_events=["mouseenter", "click"])
    listener.on_dom_event(lambda event, trial_id=simulation_id: show_trial(trial_id))
    previews[simulation_id], hover_events[simulation_id] = card, listener
    loaded_selector.options = [(f"Trial {trial_id}", trial_id) for trial_id in trials]
    grid.children = list(previews.values())
    remove_button.disabled = False
    load_button.disabled = len(trials) == MAX_PLOTS
    status.value = f"{len(trials)} / {MAX_PLOTS} trials loaded."
    show_trial(simulation_id)


def remove_trial(button):
    global active_id
    simulation_id = loaded_selector.value
    if simulation_id is None:
        return
    del trials[simulation_id]
    card = previews.pop(simulation_id)
    hover_events.pop(simulation_id).close()
    for figure in card.children[1].children:
        for mark in figure.marks:
            mark.close()
        for axis in figure.axes:
            axis.scale.close()
            axis.close()
        figure.close()
    card.children[1].close()
    card.children[0].close()
    card.close()
    grid.children = list(previews.values())
    active_id = None
    loaded_selector.options = [(f"Trial {trial_id}", trial_id) for trial_id in trials]
    remove_button.disabled = not trials
    load_button.disabled = False
    status.value = f"{len(trials)} / {MAX_PLOTS} trials loaded."
    if trials:
        show_trial(loaded_selector.value)
    else:
        for figure in focus_plot.children:
            figure.marks[0].x = []
            figure.marks[0].y = []
        parameter_bars.x, parameter_bars.y, parameter_bars.color = [], [], []
        focus_title.value = "<b>Load a trial to inspect it.</b>"


parameter_bars.on_hover(show_parameter)
load_button.on_click(lambda button: add_trial(trial_input.value))
remove_button.on_click(remove_trial)
loaded_selector.observe(lambda change: show_trial(change["new"]) if change["new"] is not None else None,
                        names="value")
display(widgets.VBox([
    widgets.HBox([trial_input, load_button, loaded_selector, remove_button],
                 layout=widgets.Layout(flex_flow="row wrap")),
    status, grid, focus_title,
    widgets.HBox([focus_plot, parameter_panel], layout=widgets.Layout(width="100%")),
]))
for simulation_id in TRIAL_IDS:
    add_trial(simulation_id)